[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/research_assistant/crewai.ipynb)

# Research assistant — CrewAI Flow

Five `@listen` stages preserve the matched research workflow and four-call budget. The shared provider supplies deterministic decisions; CrewAI supplies orchestration. Mock runtime is about one minute on CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.2", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import contextlib
import io
import json
import os
import tempfile
from pathlib import Path
from typing import ClassVar

import appdirs

os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")
appdirs.user_data_dir = lambda *args, **kwargs: str(
    Path(tempfile.gettempdir()) / "agentic-ai-tutorial-crewai"
)
from crewai.flow.flow import Flow, listen, start  # noqa: E402
from pydantic import BaseModel, ConfigDict, Field  # noqa: E402

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.safety import SafetyEngine, SafetyPolicy, UntrustedContent  # noqa: E402
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture = json.loads((ROOT / "data/research_assistant/case_mock.json").read_text())

## Crew roles and flow
Planner, evidence reviewer, writer and critic responsibilities are explicit methods. Passages are isolated as untrusted data; only validated source IDs enter the ledger and citations.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: str


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: str


Ledger.model_rebuild(_types_namespace={"Claim": Claim})


def fresh_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def search(query):
    terms = set(query.casefold().split())
    return [
        r
        for r in catalogue["records"]
        if terms & set((r["title"] + " " + r["passage"]).casefold().split())
    ]


class ResearchFlow(Flow):
    @start()
    async def planner(self):
        self.client = fresh_model()
        plan = await propose(self.client, QueryPlan, "Plan bounded searches.")
        return {"plan": plan, "trace": [{"event": "plan"}], "model_calls": 1}

    @listen(planner)
    def evidence_reviewer(self, state):
        retrieved = {r["source_id"]: r for q in state["plan"].queries for r in search(q)}
        safety = SafetyEngine(SafetyPolicy())
        assessments = {
            sid: safety.inspect_retrieved_content(
                UntrustedContent(source_id=sid, text=r["passage"])
            )
            for sid, r in retrieved.items()
        }
        safe = {sid: r for sid, r in retrieved.items() if not assessments[sid].indicators}
        return {
            **state,
            "safe": safe,
            "trace": [
                *state["trace"],
                {
                    "event": "retrieve",
                    "isolated": [sid for sid, a in assessments.items() if a.indicators],
                },
            ],
        }

    @listen(evidence_reviewer)
    async def claim_ledger(self, state):
        ledger = await propose(self.client, Ledger, f"Extract only from {state['safe']}")
        claims = tuple(c for c in ledger.claims if c.source_id in state["safe"])
        return {
            **state,
            "claims": claims,
            "model_calls": 2,
            "trace": [
                *state["trace"],
                {
                    "event": "ledger",
                    "conflicts": [c.source_id for c in claims if c.stance == "conflicting"],
                },
            ],
        }

    @listen(claim_ledger)
    async def writer(self, state):
        answer = await propose(
            self.client, Synthesis, f"S synthesise conditions and conflicts from {state['claims']}"
        )
        return {
            **state,
            "answer": answer,
            "model_calls": 3,
            "trace": [*state["trace"], {"event": "synthesis"}],
        }

    @listen(writer)
    async def critic(self, state):
        decision = await propose(
            self.client, CriticDecision, f"Check {state['answer'].model_dump()}"
        )
        valid = set(state["answer"].source_ids).issubset(state["safe"])
        return {
            **state,
            "outcome": state["answer"].outcome if decision.accepted and valid else "abstention",
            "model_calls": 4,
            "termination": "criteria_met",
            "trace": [
                *state["trace"],
                {"event": "critic", "accepted": decision.accepted, "citations_valid": valid},
            ],
        }


async def run_flow():
    with contextlib.redirect_stdout(io.StringIO()):
        return await ResearchFlow().kickoff_async()


first = await run_flow()
second = await run_flow()
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["answer"].source_ids,
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "flow_stages": 5},
    "fallback": "invalid evidence yields abstention",
}

## Limitation
Crew roles improve readability but add runtime and telemetry/storage concerns; deterministic mock runs do not measure real multi-agent disagreement.